# Campus GAT V23 - Cluster Trunk Road

건물 하나하나에 차도를 붙이는 방식이 아니라 건물군을 묶고 공통 trunk road를 학습하도록 수정한 버전입니다.


In [11]:
import os
!git -C swe3032 pull 2>/dev/null || git clone https://github.com/SKKUAIProjectTeam1/swe3032.git
os.chdir('swe3032')

Cloning into 'swe3032'...
remote: Enumerating objects: 1174, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 1174 (delta 12), reused 15 (delta 5), pack-reused 1150 (from 1)
Receiving objects: 100% (1174/1174), 188.88 MiB | 26.22 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Updating files: 100% (981/981), done.


In [ ]:
import glob, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy.ndimage import distance_transform_edt, maximum_filter, gaussian_filter
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra as sp_dijkstra

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────────────
RES           = 100
N             = RES * RES
NODE_DIM      = 5
GAT_HEADS     = 4
DIFF_STEPS    = 40
EPOCHS        = 100
LR            = 3e-4
WARMUP_EPOCHS = 30
ATTN_DROPOUT  = 0.1
ACCUM_STEPS   = 10
N_GATES       = 2
GATE_MIN_DIST = 14
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# V23 핵심 파라미터
CLUSTER_EPS       = 13.0
ROAD_CLEAR_MIN    = 3.0
ROAD_CLEAR_MAX    = 24.0
RIDGE_FILTER_SIZE = 7
ACCESS_RADIUS     = 8.0

IMG_DIR = 'collegemap/images'
TXT_DIR = 'collegemap/txt'
OUT_DIR = 'output'
CKPT    = os.path.join(OUT_DIR, 'v23_cluster_trunk.pth')
os.makedirs(OUT_DIR, exist_ok=True)

In [14]:
# ── 고정 그래프 토폴로지 ───────────────────────────────────────────────────────
def _build_edges(res):
    edges = []
    for y in range(res):
        for x in range(res):
            s = y * res + x
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    if dy == 0 and dx == 0: continue
                    nx_, ny_ = x + dx, y + dy
                    if 0 <= nx_ < res and 0 <= ny_ < res:
                        edges.append((s, ny_ * res + nx_))
    edges += [(i, i) for i in range(res * res)]
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

EDGE_INDEX = _build_edges(RES).to(DEVICE)

_bdy = []
for i in range(10, RES - 10):
    _bdy += [i, (RES-1)*RES + i, i*RES, i*RES + (RES-1)]
BOUNDARY   = sorted(set(_bdy))
BOUNDARY_T = torch.tensor(BOUNDARY, dtype=torch.long).to(DEVICE)

In [ ]:
# ── 데이터 로딩 ────────────────────────────────────────────────────────────────
def _find_txt(img_path):
    stem   = os.path.basename(img_path).replace('_building_mask.png', '')
    direct = os.path.join(TXT_DIR, stem + '_building_places.txt')
    if os.path.exists(direct): return direct
    prefix = stem.replace('-', '_').split('_')[0]
    for fn in os.listdir(TXT_DIR):
        if fn.endswith('_building_places.txt') and fn.startswith(prefix):
            return os.path.join(TXT_DIR, fn)
    return None

def _nearest_road_node(cy, cx, is_bld_grid):
    non_bld_yx = np.argwhere(is_bld_grid == 0)
    if len(non_bld_yx) == 0: return int(cy) * RES + int(cx)
    dists = (non_bld_yx[:,0] - cy)**2 + (non_bld_yx[:,1] - cx)**2
    best  = non_bld_yx[np.argmin(dists)]
    return int(best[0]) * RES + int(best[1])

def _make_common_road_ridge(is_bld_grid):
    free = is_bld_grid == 0
    dist = distance_transform_edt(free)
    clear = (dist > ROAD_CLEAR_MIN) & (dist < ROAD_CLEAR_MAX)
    campus_influence = gaussian_filter(is_bld_grid.astype(np.float32), sigma=7.0)
    near_campus = campus_influence > 0.01
    local_max = dist == maximum_filter(dist, size=RIDGE_FILTER_SIZE)
    ridge = free & clear & near_campus & local_max
    ridge_score = gaussian_filter(ridge.astype(np.float32), sigma=1.2)
    if ridge_score.max() < 1e-6:
        band = free & clear & near_campus
        ridge_score = gaussian_filter(band.astype(np.float32), sigma=1.0)
    ridge_score = ridge_score / (ridge_score.max() + 1e-6)
    return ridge_score.astype(np.float32), dist.astype(np.float32)

def _cluster_centers(centers, eps=CLUSTER_EPS):
    n = len(centers); visited = [False] * n; clusters = []
    for i in range(n):
        if visited[i]: continue
        stack = [i]; visited[i] = True; cluster = []
        while stack:
            u = stack.pop(); cluster.append(u); uy, ux = centers[u]
            for v in range(n):
                if visited[v]: continue
                vy, vx = centers[v]
                if ((uy - vy) ** 2 + (ux - vx) ** 2) ** 0.5 <= eps:
                    visited[v] = True; stack.append(v)
        clusters.append(cluster)
    return clusters

def _nearest_ridge_node(cy, cx, is_bld_grid, ridge_grid):
    candidates = np.argwhere((is_bld_grid == 0) & (ridge_grid > 0.05))
    if len(candidates) == 0:
        return _nearest_road_node(cy, cx, is_bld_grid)
    dists = (candidates[:,0] - cy)**2 + (candidates[:,1] - cx)**2
    best = candidates[np.argmin(dists)]
    return int(best[0]) * RES + int(best[1])

def _compute_gt_path_nodes(cluster_node_list, is_bld_grid, ridge_grid):
    """클러스터 대표점들의 MST를 ridge-following Dijkstra로 계산합니다.

    결과: MST 경로 위 노드들의 boolean mask.
    fallback 캠퍼스(클러스터 수 > 10)는 빈 mask 반환.
    """
    n_c = len(cluster_node_list)
    if n_c <= 1 or n_c > 10:
        return torch.zeros(N, dtype=torch.bool, device=DEVICE)

    # ridge 위를 선호하는 이동 비용 (건물은 사실상 통과 불가)
    cost = (1.0 / (ridge_grid.flatten() + 0.1)).astype(np.float32)
    cost[is_bld_grid.flatten() > 0] = 1e6

    # 벡터화된 희소 인접행렬 구성
    y_c = np.repeat(np.arange(RES), RES)
    x_c = np.tile(np.arange(RES), RES)
    rows, cols, vals = [], [], []
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0: continue
            ny, nx = y_c + dy, x_c + dx
            valid = (ny >= 0) & (ny < RES) & (nx >= 0) & (nx < RES)
            u = y_c[valid] * RES + x_c[valid]
            v = ny[valid] * RES + nx[valid]
            w = cost[v] * (1.4142 if abs(dy) + abs(dx) == 2 else 1.0)
            rows.append(u); cols.append(v); vals.append(w)
    graph = csr_matrix(
        (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
        shape=(N, N)
    )

    idx = np.array(cluster_node_list, dtype=np.int32)
    dist, prev = sp_dijkstra(graph, directed=True, indices=idx, return_predecessors=True)

    # Prim's MST
    in_mst = [False] * n_c; in_mst[0] = True; mst_edges = []
    for _ in range(n_c - 1):
        bc, bi, bj = np.inf, -1, -1
        for i in range(n_c):
            if not in_mst[i]: continue
            for j in range(n_c):
                if in_mst[j]: continue
                if dist[i, cluster_node_list[j]] < bc:
                    bc, bi, bj = dist[i, cluster_node_list[j]], i, j
        if bi == -1: break
        in_mst[bj] = True; mst_edges.append((bi, bj))

    # 경로 추적
    path_nodes = set()
    for i, j in mst_edges:
        cur = cluster_node_list[j]; src = cluster_node_list[i]
        while cur != src and cur >= 0:
            path_nodes.add(cur); cur = int(prev[i, cur])
        path_nodes.add(src)

    gt_mask = torch.zeros(N, dtype=torch.bool, device=DEVICE)
    if path_nodes:
        gt_mask[torch.tensor(list(path_nodes), dtype=torch.long)] = True
    return gt_mask

def load_campus(img_path, txt_path):
    img    = Image.open(img_path).convert('L')
    W, H   = img.size
    is_bld = (np.array(img.resize((RES,RES), resample=Image.NEAREST)) > 128).astype(np.float32)

    ridge, dist = _make_common_road_ridge(is_bld)
    dist_n = (dist / (dist.max() + 1e-6)).astype(np.float32)

    xs = np.tile(np.arange(RES), RES).astype(np.float32) / RES
    ys = np.repeat(np.arange(RES), RES).astype(np.float32) / RES
    node_feats = np.stack([is_bld.flatten(), ridge.flatten(), dist_n.flatten(), xs, ys], axis=1)

    ib_flat  = torch.tensor(is_bld.flatten(), dtype=torch.float32).to(DEVICE)
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    bld_mask = (ib_flat[src] > 0) | (ib_flat[dst] > 0)

    ns = {}
    exec(open(txt_path, encoding='utf-8').read(), ns)
    poly = ns['BUILDING_POLY']

    building_ids = list(poly.keys())
    centers = []
    for bid in building_ids:
        pts = poly[bid]
        cy = np.mean([p[1] for p in pts]) * RES / H
        cx = np.mean([p[0] for p in pts]) * RES / W
        centers.append((cy, cx))

    bld_nodes = [_nearest_road_node(cy, cx, is_bld) for cy, cx in centers]

    clusters = _cluster_centers(centers, eps=CLUSTER_EPS)
    cluster_nodes = []
    cluster_groups = []
    for cluster in clusters:
        cy = np.mean([centers[i][0] for i in cluster])
        cx = np.mean([centers[i][1] for i in cluster])
        node = _nearest_ridge_node(cy, cx, is_bld, ridge)
        if node not in cluster_nodes:
            cluster_nodes.append(node)
            cluster_groups.append([building_ids[i] for i in cluster])

    if len(cluster_nodes) <= 1 and len(bld_nodes) > 1:
        cluster_nodes = list(dict.fromkeys(bld_nodes))
        cluster_groups = [[bid] for bid in building_ids]

    # GT 경로: 클러스터 간 MST ridge-following 경로 (fallback 캠퍼스는 빈 mask)
    gt_nodes_mask = _compute_gt_path_nodes(cluster_nodes, is_bld, ridge)

    return {
        'node_feats':     torch.tensor(node_feats, dtype=torch.float32).to(DEVICE),
        'is_building':    ib_flat,
        'bld_mask':       bld_mask,
        'ridge':          torch.tensor(ridge.flatten(), dtype=torch.float32).to(DEVICE),
        'building_nodes': torch.tensor(bld_nodes, dtype=torch.long).to(DEVICE),
        'cluster_nodes':  torch.tensor(cluster_nodes, dtype=torch.long).to(DEVICE),
        'cluster_groups': cluster_groups,
        'gt_nodes_mask':  gt_nodes_mask,
        'poly': poly,
        'name': os.path.basename(img_path).replace('_building_mask.png',''),
        'W': W,
        'H': H,
    }

campuses = []
for img_path in sorted(glob.glob(os.path.join(IMG_DIR, '*_building_mask.png'))):
    txt = _find_txt(img_path)
    if txt:
        try:
            campuses.append(load_campus(img_path, txt))
        except Exception as e:
            print(f'skip {os.path.basename(img_path)}: {e}')
print(f'{len(campuses)}개 캠퍼스 로드 완료')
for c in campuses[:3]:
    n_gt = int(c['gt_nodes_mask'].sum())
    print(f"{c['name']}: buildings={len(c['building_nodes'])}, clusters={len(c['cluster_nodes'])}, gt_nodes={n_gt}")

In [16]:
# ── 모델 ──────────────────────────────────────────────────────────────────────
class MHGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.0):
        super().__init__()
        assert out_dim % heads == 0
        self.heads = heads; self.dh = out_dim // heads
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.a_s  = nn.Parameter(torch.empty(heads, self.dh))
        self.a_d  = nn.Parameter(torch.empty(heads, self.dh))
        self.norm = nn.LayerNorm(out_dim)
        self.skip = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight, gain=1.414)
        nn.init.xavier_normal_(self.a_s.unsqueeze(0)); nn.init.xavier_normal_(self.a_d.unsqueeze(0))

    def forward(self, x, ei):
        n = x.size(0); src, dst = ei[0], ei[1]
        h = self.W(x).view(n, self.heads, self.dh)
        e = F.leaky_relu((h[src]*self.a_s).sum(-1) + (h[dst]*self.a_d).sum(-1), 0.2)
        e_exp = torch.exp(e - e.max())
        denom = torch.zeros(n, self.heads, device=x.device)
        denom.scatter_add_(0, src.unsqueeze(1).expand(-1, self.heads), e_exp)
        alpha = self.drop(e_exp / (denom[src] + 1e-8))
        msg   = alpha.unsqueeze(-1) * h[dst]
        agg   = torch.zeros(n, self.heads, self.dh, device=x.device)
        agg.scatter_add_(0, src[:,None,None].expand_as(msg), msg)
        return self.norm(F.elu(agg.view(n,-1)) + self.skip(x))

class CampusGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.gat1 = MHGATLayer(NODE_DIM, 64, GAT_HEADS, ATTN_DROPOUT)
        self.gat2 = MHGATLayer(64, 64, GAT_HEADS, ATTN_DROPOUT)
        self.gat3 = MHGATLayer(64, 32, GAT_HEADS, ATTN_DROPOUT)
        self.edge_head = nn.Sequential(nn.Linear(128,32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.gate_head = nn.Linear(32, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
        # edge는 처음부터 과도하게 깔리지 않도록 낮게 시작합니다.
        nn.init.constant_(self.edge_head[-1].bias, -2.0)
        # boundary 개수가 많아서 gate sum loss가 폭발하지 않도록 sigmoid(-5)≈0.0067로 시작합니다.
        nn.init.constant_(self.gate_head.bias, -5.0)

    def forward(self, feats, ei, bld_mask):
        h1 = self.gat1(feats, ei); h2 = self.gat2(h1, ei); h3 = self.gat3(h2, ei)
        src, dst = ei[0], ei[1]
        ew = torch.sigmoid(self.edge_head(torch.cat([h2[src],h2[dst]],1)).squeeze(-1)) * (~bld_mask).float()
        gs = torch.sigmoid(self.gate_head(h3[BOUNDARY_T]).squeeze(-1))
        return ew, gs

model = CampusGAT().to(DEVICE)
print(f'파라미터: {sum(p.numel() for p in model.parameters()):,}개')


파라미터: 13,666개


In [ ]:
# ── 손실 함수 ──────────────────────────────────────────────────────────────────
def diffusion_conn_loss(ew, target_nodes, T=DIFF_STEPS, weight=1.0):
    """클러스터 대표점들이 하나의 차도망으로 연결되도록 유도합니다."""
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    n_t = target_nodes.size(0)
    if n_t <= 1:
        return ew.sum() * 0.

    init = torch.zeros(N, n_t, device=DEVICE)
    init[target_nodes, torch.arange(n_t, device=DEVICE)] = 1.0
    signals = init.clone()

    for _ in range(T):
        msg = torch.zeros(N, n_t, device=DEVICE)
        msg.scatter_add_(0, dst.unsqueeze(1).expand(-1, n_t), ew.unsqueeze(1) * signals[src])
        signals = torch.maximum(init, torch.clamp(signals + msg * 0.5, 0.0, 1.0))

    received = signals[target_nodes]
    off_diag = 1. - torch.eye(n_t, device=DEVICE)
    return F.relu(1. - received).mul(off_diag).mean() * weight

def building_access_loss(ew, building_nodes, radius=ACCESS_RADIUS, weight=1.0):
    """모든 건물 근처에 도로가 있도록 유도합니다. [B, N] 행렬 연산."""
    if building_nodes.numel() == 0:
        return ew.sum() * 0.

    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    strength = torch.zeros(N, device=DEVICE)
    strength.scatter_add_(0, src, ew)
    strength.scatter_add_(0, dst, ew)
    road_prob = torch.sigmoid(4.0 * (strength - 0.6))

    yy = (torch.arange(N, device=DEVICE) // RES).float()
    xx = (torch.arange(N, device=DEVICE) % RES).float()
    by = (building_nodes // RES).float()
    bx = (building_nodes % RES).float()

    d = torch.sqrt((yy.unsqueeze(0) - by.unsqueeze(1))**2 + (xx.unsqueeze(0) - bx.unsqueeze(1))**2 + 1e-6)
    near = torch.exp(-d / radius)
    access = (road_prob.unsqueeze(0) * near).max(dim=1).values
    return F.relu(1.0 - access).mean() * weight

def gt_road_loss(ew, gt_nodes_mask, weight=1.0):
    """GT 경로(클러스터 간 MST) 위 노드들은 반드시 강한 도로를 가져야 합니다.

    GT가 없는 캠퍼스(mask가 전부 False)는 자동으로 0 반환.
    """
    if not gt_nodes_mask.any():
        return ew.sum() * 0.

    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    strength = torch.zeros(N, device=DEVICE)
    strength.scatter_add_(0, src, ew)
    strength.scatter_add_(0, dst, ew)
    path_strength = strength[gt_nodes_mask]
    return F.relu(1.0 - path_strength).mean() * weight

def deadend_loss(ew, target_nodes=None, weight=1.0):
    """짧은 찌꺼기 지선을 줄입니다."""
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    strength = torch.zeros(N, device=DEVICE)
    strength.scatter_add_(0, src, ew)
    strength.scatter_add_(0, dst, ew)

    road_prob = torch.sigmoid(4.0 * (strength - 0.6))
    dead_prob = road_prob * torch.sigmoid(5.0 * (1.6 - strength))

    if target_nodes is not None and target_nodes.numel() > 0:
        exempt = torch.zeros(N, device=DEVICE)
        exempt[target_nodes] = 1.0
        for _ in range(2):
            msg = torch.zeros(N, device=DEVICE)
            msg.scatter_add_(0, dst, exempt[src])
            exempt = torch.clamp(exempt + msg, 0.0, 1.0)
        dead_prob = dead_prob * (1.0 - exempt)

    return dead_prob.mean() * weight

def loss_terms(ew, gs, c, scale):
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    rs = (c['ridge'][src] + c['ridge'][dst]) * 0.5

    ridge_loss = (ew * (1. - rs)).mean() * (scale * 3.0 + 0.1)
    ridge_keep = -(ew * rs).mean() * (scale * 0.35)

    # GT가 방향을 잡아주므로 conn은 보조 역할로 weight 조정
    conn   = diffusion_conn_loss(ew, c['cluster_nodes'], weight=scale * 15.0 + 1.0)
    access = building_access_loss(ew, c['building_nodes'], radius=ACCESS_RADIUS, weight=scale * 2.0 + 0.2)
    gt     = gt_road_loss(ew, c['gt_nodes_mask'], weight=scale * 25.0 + 2.0)

    sparse = ew.mean() * 1.6

    strength = torch.zeros(N, device=DEVICE)
    strength.scatter_add_(0, src, ew)
    strength.scatter_add_(0, dst, ew)
    degree = F.relu(strength - 4.).mean() * 150.

    dead = deadend_loss(ew, c['cluster_nodes'], weight=scale * 2.0)

    gate_count = (gs.sum() - N_GATES).pow(2) * 2.0
    gate_sharp = (gs * (1.0 - gs)).mean() * 0.05
    gate = gate_count + gate_sharp

    return {
        'ridge': ridge_loss,
        'ridge_keep': ridge_keep,
        'conn': conn,
        'access': access,
        'gt': gt,
        'sparse': sparse,
        'degree': degree,
        'dead': dead,
        'gate': gate,
    }

def loss_fn(ew, gs, c, scale):
    terms = loss_terms(ew, gs, c, scale)
    return sum(terms.values())

In [18]:
# ── 학습 ──────────────────────────────────────────────────────────────────────
opt    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=5e-6)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
n = len(campuses); loss_curve = []

for ep in range(1, EPOCHS+1):
    model.train(); scale = min(1.0, ep/WARMUP_EPOCHS); total = 0.; opt.zero_grad()
    debug_terms = None

    for i, c in enumerate(campuses):
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            ew, gs = model(c['node_feats'], EDGE_INDEX, c['bld_mask'])
            loss   = loss_fn(ew, gs, c, scale) / n
        scaler.scale(loss).backward(); total += loss.item() * n

        if i == 0 and (ep % 25 == 0 or ep == 1):
            with torch.no_grad():
                debug_terms = {k: float(v.detach().cpu()) for k, v in loss_terms(ew, gs, c, scale).items()}

        if (i+1) % ACCUM_STEPS == 0 or (i+1) == n:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    sch.step(); loss_curve.append(total/n)
    if ep % 1 == 0 or ep == 1:
        msg = f'[{ep:4d}/{EPOCHS}] loss={total/n:.4f}  lr={sch.get_last_lr()[0]:.2e}'
        if debug_terms:
            brief = ' '.join([f'{k}={v:.3f}' for k, v in debug_terms.items()])
            msg += '  | ' + brief
        print(msg)

torch.save(model.state_dict(), CKPT)
print(f'저장: {CKPT}')


/tmp/ipykernel_1996/2202030108.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_1996/2202030108.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


[   1/300] loss=2.7811  lr=3.00e-04  | ridge=0.015 ridge_keep=-0.000 conn=1.733 access=0.015 sparse=0.155 degree=0.000 dead=0.009 gate=1.832
[   2/300] loss=2.4432  lr=3.00e-04
[   3/300] loss=2.6775  lr=3.00e-04
[   4/300] loss=3.0279  lr=3.00e-04
[   5/300] loss=3.3875  lr=3.00e-04
[   6/300] loss=3.7592  lr=3.00e-04
[   7/300] loss=4.1346  lr=3.00e-04
[   8/300] loss=4.5105  lr=2.99e-04
[   9/300] loss=4.8845  lr=2.99e-04
[  10/300] loss=5.2587  lr=2.99e-04
[  11/300] loss=5.6310  lr=2.99e-04
[  12/300] loss=6.0028  lr=2.99e-04
[  13/300] loss=6.3750  lr=2.99e-04
[  14/300] loss=6.7462  lr=2.98e-04
[  15/300] loss=7.1173  lr=2.98e-04
[  16/300] loss=7.4878  lr=2.98e-04
[  17/300] loss=7.8560  lr=2.98e-04
[  18/300] loss=8.2275  lr=2.97e-04
[  19/300] loss=8.5949  lr=2.97e-04
[  20/300] loss=8.9574  lr=2.97e-04
[  21/300] loss=9.3228  lr=2.96e-04
[  22/300] loss=9.6849  lr=2.96e-04
[  23/300] loss=10.0527  lr=2.96e-04
[  24/300] loss=10.4101  lr=2.95e-04
[  25/300] loss=10.7751  lr=2

In [ ]:
# ── 손실 커브 & 시각화 ─────────────────────────────────────────────────────────
plt.figure(figsize=(9,3)); plt.plot(loss_curve); plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

def _draw_node(ax, node, color, label=None, size=55):
    y, x = divmod(int(node), RES)
    ax.scatter(x, y, s=size, c=color, edgecolors='white', linewidths=0.5, zorder=9)
    if label:
        ax.text(x, y-1.5, label, color='white', ha='center', va='bottom', fontsize=5.5, zorder=10)

def plot_campus(c, show_clusters=True, show_ridge=False, show_gt=True):
    model.eval()
    with torch.no_grad():
        ew, gs = model(c['node_feats'], EDGE_INDEX, c['bld_mask'])
    ew_np=ew.cpu().numpy(); gs_np=gs.cpu().numpy(); ei=EDGE_INDEX.cpu(); ib_np=c['is_building'].cpu().numpy()
    W,H = c['W'],c['H']

    order=np.argsort(gs_np)[::-1]; chosen=[]
    for gi in order:
        gx=BOUNDARY[gi]%RES; gy=BOUNDARY[gi]//RES
        if all(abs(gx-BOUNDARY[p]%RES)+abs(gy-BOUNDARY[p]//RES)>=GATE_MIN_DIST for p in chosen):
            chosen.append(gi)
        if len(chosen)==N_GATES: break

    fig,ax=plt.subplots(figsize=(12,12)); ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')

    if show_ridge:
        ridge_img = c['ridge'].detach().cpu().numpy().reshape(RES, RES)
        ax.imshow(ridge_img, cmap='gray', alpha=0.22, extent=[0, RES, RES, 0])

    for bid,pts in c['poly'].items():
        sc=[(p[0]*RES/W,p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc,closed=True,facecolor='#2d3436',edgecolor='#00cec9',lw=0.8,alpha=0.9))
        ax.text(np.mean([p[0] for p in sc]),np.mean([p[1] for p in sc]),bid,color='white',ha='center',va='center',fontsize=5.5,fontweight='bold')

    # GT 경로 표시 (반투명 빨간 점)
    if show_gt and c['gt_nodes_mask'].any():
        gt_np = c['gt_nodes_mask'].cpu().numpy()
        gt_idx = np.where(gt_np)[0]
        gty, gtx = gt_idx // RES, gt_idx % RES
        ax.scatter(gtx, gty, s=8, c='#ff6b6b', alpha=0.5, zorder=4, label='GT path')

    thr=np.percentile(ew_np,98.5); mxw=max(ew_np.max(),thr+1e-8)
    for i in range(len(ew_np)):
        w=ew_np[i]
        if w<thr: continue
        s,d=ei[0,i].item(),ei[1,i].item()
        if ib_np[s]>0 or ib_np[d]>0: continue
        y1,x1=divmod(s,RES); y2,x2=divmod(d,RES)
        ax.plot([x1,x2],[y1,y2],color='#ffd32a',alpha=0.88,lw=(w-thr)/(mxw-thr)*8+1,solid_capstyle='round')

    if show_clusters:
        for idx, node in enumerate(c['cluster_nodes'].detach().cpu().numpy()):
            _draw_node(ax, node, '#74b9ff', label=f'C{idx}', size=60)

    for k,gi in enumerate(chosen):
        node=BOUNDARY[gi]; gx,gy=node%RES,node//RES
        ax.scatter(gx,gy,s=500,c='#ff3838',marker='*',edgecolors='white',lw=1.5,zorder=10)
        ax.text(gx,gy-2.5,f'GATE {chr(65+k)}',color='white',ha='center',fontsize=9,fontweight='bold',bbox=dict(fc='#d63031',alpha=0.8,ec='none',pad=1))

    ax.set_xlim(0,RES); ax.set_ylim(RES,0); ax.axis('off')
    n_gt = int(c['gt_nodes_mask'].sum())
    ax.set_title(
        f'V23 Cluster Trunk · {c["name"].replace("_"," ").title()} · '
        f'buildings {len(c["building_nodes"])} → clusters {len(c["cluster_nodes"])} · gt {n_gt}nodes',
        color='white', fontsize=13, pad=10
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,f'v23_cluster_trunk_{c["name"]}.png'),dpi=150,bbox_inches='tight',facecolor='#0d1117')
    plt.show(); plt.close()

def print_clusters(c):
    print(f"{c['name']} clusters: {len(c['cluster_groups'])}")
    for i, group in enumerate(c['cluster_groups']):
        print(f"C{i:02d}: {group}")

# 특정 캠퍼스 확인
TARGET = 'sungkyunkwan_university'
target = next((c for c in campuses if TARGET in c['name']), None)
if target:
    print_clusters(target)
    plot_campus(target, show_clusters=True, show_ridge=False, show_gt=True)